# 🚀 Лабораторная работа 13. Итоговый локальный AI-ассистент

Цель: объединить RAG, Tools и Agent Controller в одну учебную систему.

Для воспроизводимости решения LLM сначала симулируются. В конце оставлено место для подключения локальной Qwen через Ollama.


# 1. Импорт и настройки

In [ ]:
import math
import re
from collections import Counter
from typing import Any, Callable

import torch

MAX_STEPS = 5
TOP_K = 2

# 2. Knowledge Base

In [ ]:
DOCUMENTS = [
    {
        "source": "ai_course.md",
        "text": (
            "RAG добавляет найденные фрагменты документов "
            "в контекст модели и не изменяет её weights."
        ),
    },
    {
        "source": "lora.md",
        "text": (
            "LoRA замораживает базовые веса модели "
            "и обучает небольшие low-rank адаптеры."
        ),
    },
    {
        "source": "agents.md",
        "text": (
            "AI-агент может выбирать инструменты, "
            "получать Observation и продолжать работу."
        ),
    },
]

# 3. Простой RAG

In [ ]:
def tokenize(text):
    return re.findall(
        r"[а-яёa-z0-9]+",
        text.lower(),
    )


vocabulary = sorted(
    {
        token
        for document in DOCUMENTS
        for token in tokenize(document["text"])
    }
)

token_to_id = {
    token: index
    for index, token in enumerate(vocabulary)
}

document_frequency = Counter()

for document in DOCUMENTS:
    for token in set(tokenize(document["text"])):
        document_frequency[token] += 1


def text_to_vector(text):
    tokens = tokenize(text)
    counts = Counter(tokens)

    vector = torch.zeros(
        len(vocabulary),
        dtype=torch.float32,
    )

    for token, count in counts.items():
        if token not in token_to_id:
            continue

        tf = count / max(len(tokens), 1)
        df = document_frequency.get(token, 0)

        idf = math.log(
            (len(DOCUMENTS) + 1)
            / (df + 1)
        ) + 1.0

        vector[token_to_id[token]] = tf * idf

    return vector


document_vectors = torch.stack(
    [
        text_to_vector(document["text"])
        for document in DOCUMENTS
    ]
)


def cosine_similarity(a, b):
    denominator = (
        torch.linalg.vector_norm(a)
        * torch.linalg.vector_norm(b)
    )

    if denominator.item() == 0:
        return 0.0

    return float(
        torch.dot(a, b) / denominator
    )


def retrieve(question, top_k=TOP_K):
    query_vector = text_to_vector(question)

    results = []

    for document, vector in zip(
        DOCUMENTS,
        document_vectors,
    ):
        score = cosine_similarity(
            query_vector,
            vector,
        )

        results.append(
            {
                **document,
                "score": score,
            }
        )

    results.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    return results[:top_k]

# 4. Проверяем Retrieval

In [ ]:
for result in retrieve("Как работает LoRA?"):
    print(result)

# 5. Создаём Tools

In [ ]:
def add_numbers(a, b):
    return {
        "status": "ok",
        "result": a + b,
    }


def word_count(text):
    return {
        "status": "ok",
        "result": len(text.split()),
    }


def search_knowledge_base(question):
    return {
        "status": "ok",
        "results": retrieve(question),
    }

# 6. Tool Registry

In [ ]:
TOOL_REGISTRY: dict[
    str,
    Callable[..., dict[str, Any]],
] = {
    "add_numbers": add_numbers,
    "word_count": word_count,
    "search_knowledge_base": search_knowledge_base,
}

print(TOOL_REGISTRY.keys())

# 7. Dispatcher

In [ ]:
def execute_tool(tool_name, arguments):
    if tool_name not in TOOL_REGISTRY:
        return {
            "status": "error",
            "message": f"Неизвестный Tool: {tool_name}",
        }

    try:
        return TOOL_REGISTRY[tool_name](**arguments)
    except Exception as error:
        return {
            "status": "error",
            "message": str(error),
        }

# 8. Симулируем решение LLM

In [ ]:
def simulated_llm_decision(user_message, observations):
    text = user_message.lower()

    if "сложи" in text:
        numbers = re.findall(
            r"-?\d+(?:\.\d+)?",
            text,
        )

        if len(numbers) >= 2:
            return {
                "type": "tool",
                "name": "add_numbers",
                "arguments": {
                    "a": float(numbers[0]),
                    "b": float(numbers[1]),
                },
            }

    if "сколько слов" in text:
        if not observations:
            return {
                "type": "rag",
                "query": user_message,
            }

        last = observations[-1]

        if last["kind"] == "rag":
            results = last["result"]["results"]

            if not results:
                return {
                    "type": "final",
                    "content": "В базе знаний ничего не найдено.",
                }

            return {
                "type": "tool",
                "name": "word_count",
                "arguments": {
                    "text": results[0]["text"],
                },
            }

        if (
            last["kind"] == "tool"
            and last["name"] == "word_count"
            and last["result"]["status"] == "ok"
        ):
            return {
                "type": "final",
                "content": (
                    "В найденном фрагменте "
                    f"{last['result']['result']} слов."
                ),
            }

    if any(
        keyword in text
        for keyword in ("rag", "lora", "агент", "документ")
    ):
        if not observations:
            return {
                "type": "rag",
                "query": user_message,
            }

        last = observations[-1]

        if last["kind"] == "rag":
            results = last["result"]["results"]

            if not results:
                return {
                    "type": "final",
                    "content": "В базе знаний ничего не найдено.",
                }

            best = results[0]

            return {
                "type": "final",
                "content": (
                    f"По базе знаний: {best['text']} "
                    f"(источник: {best['source']})"
                ),
            }

    return {
        "type": "final",
        "content": "Это прямой ответ модели без Tool и RAG.",
    }

# 9. Assistant Controller

In [ ]:
def run_assistant(user_message, max_steps=MAX_STEPS):
    observations = []

    for step in range(max_steps):
        decision = simulated_llm_decision(
            user_message,
            observations,
        )

        print(f"STEP {step + 1}:", decision)

        if decision["type"] == "final":
            return decision["content"]

        if decision["type"] == "rag":
            result = search_knowledge_base(
                decision["query"]
            )

            observation = {
                "kind": "rag",
                "result": result,
            }

            observations.append(observation)
            print("OBSERVATION:", observation)
            continue

        if decision["type"] == "tool":
            result = execute_tool(
                decision["name"],
                decision["arguments"],
            )

            observation = {
                "kind": "tool",
                "name": decision["name"],
                "result": result,
            }

            observations.append(observation)
            print("OBSERVATION:", observation)
            continue

        return "Ошибка Controller: неизвестный тип решения."

    return "Assistant остановлен: достигнут MAX_STEPS." 

# 10. Direct запрос

In [ ]:
print(
    run_assistant(
        "Объясни, что такое Tensor."
    )
)

# 11. Tool запрос

In [ ]:
print(
    run_assistant(
        "Сложи 125 и 37."
    )
)

# 12. RAG запрос

In [ ]:
print(
    run_assistant(
        "Как работает LoRA?"
    )
)

# 13. Многошаговый запрос

In [ ]:
print(
    run_assistant(
        "Сколько слов в найденной заметке про LoRA?"
    )
)

# 14. Место для локальной Qwen

In [ ]:
MODEL_NAME = "your-local-qwen-model"


def ask_qwen(messages):
    try:
        import ollama
    except ImportError:
        return {
            "status": "error",
            "message": "Python-пакет ollama не установлен.",
        }

    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=messages,
            options={
                "temperature": 0.2,
            },
        )

        return {
            "status": "ok",
            "content": response["message"]["content"],
        }

    except Exception as error:
        return {
            "status": "error",
            "message": str(error),
        }

В реальной версии `simulated_llm_decision()` заменяется вызовом локальной Qwen, которая возвращает структурированное решение: `direct`, `rag` или `tool`.


# 15. Read-only permissions

In [ ]:
READ_ONLY_TOOLS = {
    "add_numbers",
    "word_count",
    "search_knowledge_base",
}


def is_allowed(tool_name):
    return tool_name in READ_ONLY_TOOLS


print(is_allowed("search_knowledge_base"))

# 16. 📌 Что нужно запомнить

```text
LLM
+
RAG
+
Tools
+
Controller
+
Optional LoRA
=
Local AI Assistant
```


# 17. 🧩 Финальный эксперимент

Добавь новый Read-only Tool:

```text
get_text_length
```

Потом добавь его в Registry и научи Controller выбирать его для подходящего запроса.


# 18. 🏁 Курс завершён

Теперь можно переходить от учебных глав к отдельным настоящим проектам.
